# EndoScan ER — Phase 2 operator runbook (CLEAN — four stops, *not* Run-all)

**You are the operator. You never edit Python.** You paste reviewed values into the
CONFIG cells (Stop 2a, Stop 3a) and run one stop at a time, then **STOP** and send the
printed outputs to Claude Chat. Every cell calls only merged, tested code
(`endoscan_core` + the tested ER staging helpers + `pipelines/endpoints/ER/run.py`):
there is no notebook-only logic and no debugging hotfixes.

- **Preflight / Setup** — verify the read-only PAT, clone read-only, install, import,
  mount Drive + DVC.
- **Stop 1** — light inspect of metadata + CERAPP archives; expected gctx shape (no download).
- **Stop 2** — labels + identity (2a), join + per-hop counts + **hard guard** (2b),
  the heavy ~20 GB gctx download/slice/fuse (2c, unreachable unless the guard passed),
  gate-only run (2d, training impossible here).
- **Stop 3** — train (explicit human approval). **Stop 4** — DVC-push + review bundle (no GitHub push).

EndoScan is a pre-screening / prioritization tool — **no regulatory-grade claims**.

## PREFLIGHT — verify the read-only PAT (run FIRST)
Set the Colab Secret **`GH_PAT_RO`** to a fine-grained PAT scoped to the single repo
`Rirkella/endoscan-platform` with **Contents: Read-only** and a short expiry. The token
is checked here and never printed.

In [ ]:
# Preflight) verify the read-only single-repo PAT BEFORE install/clone/download.
# Defines REPO + TOKEN used by Setup. Never prints the token or the raw response.
import requests
from google.colab import userdata

REPO = 'Rirkella/endoscan-platform'
try:
    TOKEN = (userdata.get('GH_PAT_RO') or '').strip()
except Exception:
    TOKEN = ''
if not TOKEN:
    raise SystemExit(
        'Set Colab Secret GH_PAT_RO to a fine-grained PAT '
        '(single repo Rirkella/endoscan-platform, Contents: Read-only, short expiry).')

resp = requests.get(
    f'https://api.github.com/repos/{REPO}',
    headers={'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/vnd.github+json'},
    timeout=30,
)
if resp.status_code == 200:
    print('repo access OK')
else:
    raise SystemExit(
        f'repo access FAILED (HTTP {resp.status_code}). Check that GH_PAT_RO is a '
        'fine-grained PAT for this single repo with Contents: Read-only and is not expired.')

## SETUP — clone (read-only), install, import preflight, Drive + DVC
Run the three Setup cells in order. The clone is token-safe and read-only; the import
preflight confirms `endoscan_core` imports in-kernel before any data work.

In [ ]:
# Setup-clone) token-safe READ-ONLY clone + numpy-2 install (no numpy pin, no cmapPy).
import os, stat, subprocess, sys
from pathlib import Path

# rdkit is STAGING-ONLY (used in Stop 2a to derive InChIKeys from CERAPP InChI strings,
# offline + deterministic). It is installed HERE in the Colab runbook only — it is NOT a
# dependency of endoscan_core and is NOT installed in CI.
!pip -q install 'scikit-learn>=1.8' pyarrow dvc requests rdkit

url = f'https://github.com/{REPO}.git'  # tokenless URL — the PAT is supplied via GIT_ASKPASS
# Transient GIT_ASKPASS helper: reads the token from the env (NOT argv/URL/config) and
# emits a throwaway username + the token. The token never touches the URL, argv,
# .git/config, the shell, or any printed output.
askpass = Path('/tmp/gh_askpass.sh')
askpass.write_text('#!/usr/bin/env bash\ncase "$1" in\n  *Username*) echo "x-access-token";;\n  *) echo "$GH_PAT_RO";;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
clone_env = {**os.environ, 'GH_PAT_RO': TOKEN, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0'}
subprocess.run(['git', 'clone', '--depth', '1', url], check=True, env=clone_env)
subprocess.run(['git', 'remote', 'set-url', 'origin', url], cwd='endoscan-platform', check=True)
askpass.unlink(missing_ok=True)

%cd endoscan-platform
# Editable install with the SAME interpreter as the kernel (sys.executable). pip writes a
# .pth processed only at interpreter startup, so refresh the running kernel's import state.
!{sys.executable} -m pip install -q -e packages/endoscan_core
import site, importlib
site.main(); importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath('packages/endoscan_core'))
sys.path.insert(0, 'pipelines/endpoints/ER/staging')

import numpy, pandas, h5py, pyarrow
print('imports OK — numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| h5py', h5py.__version__, '| pyarrow', pyarrow.__version__)
print('Cloned read-only; origin =', url)

In [ ]:
# Setup-import) IN-KERNEL import preflight — confirm endoscan_core imports in THIS kernel
# (NOT a !python -c subprocess, which would falsely pass before the .pth refresh).
import os
from pathlib import Path

print('cwd:', os.getcwd())
if not Path('packages/endoscan_core').is_dir():
    raise SystemExit('packages/endoscan_core not found — run Setup-clone first.')
try:
    import endoscan_core  # noqa: F401 — in-kernel import
    from endoscan_core.datasets import load_sources  # noqa: F401
except ImportError as exc:
    raise SystemExit(f'endoscan_core import FAILED in-kernel ({exc}). Re-run Setup-clone.')
print('endoscan_core import OK')

In [ ]:
# Setup-drive) mount Google Drive + configure the DVC LOCAL remote (path in .dvc/config.local).
from google.colab import drive; drive.mount('/content/drive')
!dvc remote add --local er_gdrive /content/drive/MyDrive/endoscan-dvc || true
!dvc remote default --local er_gdrive
print('DVC local remote -> /content/drive/MyDrive/endoscan-dvc')

## STOP 1 — light inspect (no 20 GB gctx)
Run cell **1c**, then STOP. It fetches + inspects `sig_info`/`gene_info`/`pert_info` and
the CERAPP experimental archives, reports the landmark count and the **expected** gctx
shape from its filename — it does **not** download the 19.9 GB gctx (that is Stop 2c).

In [ ]:
# 1c) LIGHT inspect — metadata + CERAPP only. No gctx download; nothing staged or trained.
from pathlib import Path
import gzip, pandas as pd
import fetch
from endoscan_core.datasets import load_sources

loc = {l.name: l.url for s in load_sources().sources for l in s.locators}
raw = Path('data/raw'); raw.mkdir(parents=True, exist_ok=True)

small = {'sig_info': loc['gse92742_sig_info'], 'gene_info': loc['gse92742_gene_info'],
         'pert_info': loc['gse92742_pert_info']}
for name, u in small.items():
    fetch.fetch_url(u, raw / f'{name}.txt.gz')
    head = pd.read_csv(raw / f'{name}.txt.gz', sep='\t', nrows=5)
    n = sum(1 for _ in gzip.open(raw / f'{name}.txt.gz', 'rt')) - 1
    print(f'\n=== {name}: {n} rows ===\ncolumns: {list(head.columns)}'); display(head.head())

gene_info = pd.read_csv(raw / 'gene_info.txt.gz', sep='\t')
landmark_count = (int(gene_info['pr_is_lm'].astype(str).isin({'1', 'True', 'true'}).sum())
                  if 'pr_is_lm' in gene_info.columns else 'unknown (confirm GENE_LM_COL at Stop 2)')
print('\nlandmark genes (pr_is_lm set):', landmark_count)

cerapp_article_ids = [loc[k].rstrip('/').split('/')[-1]
                      for k in loc if 'cerapp' in k and 'figshare.com' in loc[k]]
gaftp_url = next((loc[k] for k in loc if 'gaftp' in k), None)
print('\nCERAPP figshare article ids:', cerapp_article_ids, '| gaftp mirror:', bool(gaftp_url))
cerapp_src = raw / 'cerapp_src'; cerapp_manifest = []
try:
    cerapp_files = fetch.fetch_cerapp_experimental(cerapp_src, cerapp_article_ids, gaftp_url=gaftp_url)
    print('Downloaded + validated CERAPP artifacts:', [p.name for p in cerapp_files])
    print('\n=== CERAPP tabular files (relative to data/raw/cerapp_src) ===')
    cerapp_manifest = fetch.inspect_cerapp_archives(cerapp_files, cerapp_src)
except Exception as exc:
    print('CERAPP RESOLVE/VALIDATE FAILED:', exc)

if not cerapp_manifest:
    raise SystemExit('STOP 1c HALTED: no readable CERAPP experimental table. Fix the source '
                     'or upload manually, then re-run 1c. The gctx is NOT downloaded until this succeeds.')

gctx_name = Path(loc['gse92742_level5_modz_gctx']).name
dims = fetch.parse_gctx_dims(gctx_name)
gctx_expected = (f'{dims[1]} genes x {dims[0]} signatures' if dims else 'unknown (filename unparsed)')
print('\n================ STOP 1 SUMMARY ================')
print('landmark genes (pr_is_lm):', landmark_count)
print('CERAPP tables found:', len(cerapp_manifest), '->', [m['rel'] for m in cerapp_manifest])
print('gctx (EXPECTED from filename, NOT downloaded):', gctx_expected)
print('gctx file:', gctx_name)
print('================================================')

### ⛔ STOP 1 — send Claude Chat
The printed **headers + row counts** for `sig_info`/`gene_info`/`pert_info`, the
**landmark count** (`pr_is_lm`), the **CERAPP table manifest** (every tabular file, with
delimiter + experimental/consensus hint), and the **EXPECTED gctx shape** (from the
filename — the gctx is downloaded in Stop 2c, not now). Wait for the confirmed column
names + chosen CERAPP table before Stop 2.

## STOP 2 — labels, identity, join, guard, then gate ONLY
Paste the reviewed values into **2a-config**, then run 2a -> 2b -> 2c -> 2d. The heavy
gctx download in **2c** is unreachable unless the **2b guard** passes. **Training cannot
happen in this stop** (approval is hard-coded False in 2d).

In [ ]:
# 2a-config) paste ONLY these values (column names Claude Chat confirmed). No logic.
SIG_ID_COL     = ''   # sig_info: signature id
SIG_PERT_COL   = ''   # sig_info: perturbagen/compound id
SIG_CELL_COL   = ''   # sig_info: cell line
SIG_DOSE_COL   = ''   # sig_info: dose
SIG_TIME_COL   = ''   # sig_info: time
GENE_LM_COL    = ''   # gene_info: landmark flag column
GENE_ID_COL    = ''   # gene_info: gene id
GENE_SYM_COL   = ''   # gene_info: gene symbol
PERT_ID_COL    = ''   # pert_info: perturbagen id
PERT_INCHI_COL = ''   # pert_info: FULL 27-char inchi_key (NOT the 14-char prefix); -666 skipped
CERAPP_TABLE        = ''  # cerapp: chosen EXPERIMENTAL table rel-path under data/raw/cerapp_src
CERAPP_LABEL_MODE      = 'binding'           # 'binding' (assay-class) | 'agonist' | 'antagonist' | 'any'
CERAPP_MODE_COL        = 'Mode'              # cerapp: CERAPP endpoint column (agonist/antagonist axis)
CERAPP_CASRN_COL       = 'CASRN'             # cerapp: CASRN column
CERAPP_ASSAY_CLASS_COL = 'ASSAY_CLASS_NAME'  # cerapp: assay-class column (binding filter)
CERAPP_ACTIVE_COL      = 'All_active'        # cerapp: 0/1 active-call column
CERAPP_INCHI_COL       = 'InChI_Code'        # cerapp: InChI string (identity source)

In [ ]:
# 2a) CERAPP labels -> ONE ER label/compound, then derive a full InChIKey per compound.
import importlib.util, sys, pandas as pd
from pathlib import Path
import cerapp, identity, pubchem, stage_er, fetch
from endoscan_core.datasets import load_sources

raw = Path('data/raw'); staged = Path('data/staged/er'); staged.mkdir(parents=True, exist_ok=True)
loc = {l.name: l.url for s in load_sources().sources for l in s.locators}
norm = identity.normalize_inchikey  # strip+UPPERCASE; None for ''/NaN/-666 sentinels

# Gate minimum from the registry (single source of truth) — drives the 2b/2c guard.
spec = importlib.util.spec_from_file_location('er_run', 'pipelines/endpoints/ER/run.py')
er_run = importlib.util.module_from_spec(spec); sys.modules['er_run'] = er_run; spec.loader.exec_module(er_run)
MIN_OVERLAP = er_run.load_thresholds(Path('registry/data/quality_gates.yaml')).min_overlap

cerapp_table = (raw / 'cerapp_src' / CERAPP_TABLE) if CERAPP_TABLE else (raw / 'cerapp_experimental.csv')
_read = fetch.read_tabular(cerapp_table)
assert _read is not None, f'CERAPP table not readable: {cerapp_table} (check CERAPP_TABLE)'
labels, conflicts = cerapp.assemble_evaluation_labels(
    _read[0].to_dict('records'), casrn_col=CERAPP_CASRN_COL, assay_class_col=CERAPP_ASSAY_CLASS_COL,
    active_col=CERAPP_ACTIVE_COL, inchi_col=CERAPP_INCHI_COL, mode_col=CERAPP_MODE_COL,
    label_mode=CERAPP_LABEL_MODE)

def _casrn_resolver(casrn):  # fallback only, for rows with no usable InChI
    rows, _ = fetch.pubchem_mapping_with_stats([casrn]); m = pubchem.normalize_mapping(rows)
    return m[0]['inchikey'] if m else None
labels, id_stats = identity.derive_inchikeys(labels, casrn_resolver=_casrn_resolver)
pd.DataFrame(labels).to_csv(staged / 'cerapp.csv', index=False)  # carries the derived InChIKey
pd.DataFrame([
    {'input_id': r['casrn'], 'input_id_type': 'CASRN', 'inchikey': r['inchikey'],
     'mapping_confidence': r['inchikey_source']} for r in labels if r.get('inchikey')
]).to_csv(staged / 'pubchem.csv', index=False)  # CASRN -> derived InChIKey (the join identity)

_resolved = id_stats['derived_from_inchi'] + id_stats['derived_from_casrn_fallback']
print('CERAPP labels     :', len(labels), '|', len(conflicts), 'conflicts EXCLUDED')
print('identity resolved :', _resolved, '/', id_stats['n_labels'])
print('  from_inchi / casrn_fallback / unresolved:',
      id_stats['derived_from_inchi'], '/', id_stats['derived_from_casrn_fallback'], '/', id_stats['unresolved'])

In [ ]:
# 2b) Join CERAPP InChIKeys to LINCS, filter MCF7/A549 -> per-hop counts + HARD GUARD.
labelled_inchikeys = {norm(r['inchikey']) for r in labels if r.get('inchikey')}
labelled_inchikeys.discard(None)
inchi_label = {}
for r in labels:
    _k = norm(r.get('inchikey'))
    if _k is not None:
        inchi_label[_k] = r['label']

pert_info = pd.read_csv(raw / 'pert_info.txt.gz', sep='\t')
pert_to_inchi = {}
for pid, ikey in zip(pert_info[PERT_ID_COL].astype(str), pert_info[PERT_INCHI_COL]):
    nk = norm(ikey)  # full inchi_key, NOT the 14-char prefix; -666 -> None (skipped)
    if nk is not None:
        pert_to_inchi[pid] = nk
lincs_inchikeys = set(pert_to_inchi.values())
matched_inchikeys = labelled_inchikeys & lincs_inchikeys
matched_pert_ids = {pid for pid, k in pert_to_inchi.items() if k in matched_inchikeys}

sig_info = pd.read_csv(raw / 'sig_info.txt.gz', sep='\t')
sig_meta = stage_er.assemble_sig_meta(sig_info, pert_to_inchi, labelled_inchikeys,
    sig_id_col=SIG_ID_COL, pert_id_col=SIG_PERT_COL, cell_id_col=SIG_CELL_COL,
    dose_col=SIG_DOSE_COL, time_col=SIG_TIME_COL)
n_sig_ids = sig_meta['sig_id'].nunique()
overlap_inchikeys = set(sig_meta.iloc[:, 0])  # col 0 = compound identity (assemble_sig_meta contract)
overlap = len(overlap_inchikeys)
n_pos = sum(1 for k in overlap_inchikeys if inchi_label.get(k) == 1)
n_neg = sum(1 for k in overlap_inchikeys if inchi_label.get(k) == 0)

print('matched LINCS InChIKeys   :', len(matched_inchikeys))
print('matched pert_ids          :', len(matched_pert_ids))
print('selected MCF7/A549 sig_ids:', n_sig_ids)
print('pre-heavy overlap         :', overlap, '(positives=%d, negatives=%d)' % (n_pos, n_neg))
def _ex(keys): return [f'{k!r} (len {len(k)})' for k in sorted(keys)[:3]]
print('key-format CERAPP :', _ex(labelled_inchikeys) or '(none)', '(full=27 vs prefix=14)')
print('key-format LINCS  :', _ex(lincs_inchikeys) or '(none)')

# HARD GUARD — the heavy gctx step (2c) is unreachable unless ALL of these hold.
if not (n_sig_ids > 0 and overlap >= MIN_OVERLAP and n_pos > 0 and n_neg > 0):
    _hops = [('hop1 labels', len(labels)), ('hop2 identity', _resolved),
             ('hop3 labelled', len(labelled_inchikeys)), ('hop4 LINCS', len(lincs_inchikeys)),
             ('hop5 matched', len(matched_inchikeys)), ('hop6 pert_ids', len(matched_pert_ids)),
             ('hop7 sig_ids', n_sig_ids), ('hop8 overlap', overlap)]
    _last = next((nm for nm, c in reversed(_hops) if c > 0), 'none')
    raise SystemExit(
        f'STOP — NO gctx download. Guard failed (need sig_ids>0, overlap>={MIN_OVERLAP}, pos>0, '
        f'neg>0; got sig_ids={n_sig_ids}, overlap={overlap}, pos={n_pos}, neg={n_neg}). '
        f'Last non-zero hop = {_last}. Fix the join/identity, NOT the download.')
print(f'GUARD PASSED: sig_ids={n_sig_ids}, overlap={overlap} >= {MIN_OVERLAP}, pos={n_pos}, neg={n_neg}')

In [ ]:
# 2c) HEAVY (~20-40 min). RE-ASSERT the 2b guard, THEN download + slice + fuse -> lincs.parquet.
assert n_sig_ids > 0 and overlap >= MIN_OVERLAP and n_pos > 0 and n_neg > 0, (
    'guard not satisfied — run cell 2b first; the gctx download must not run')  # defense in depth
gene_info = pd.read_csv(raw / 'gene_info.txt.gz', sep='\t')
gene_ids, gene_syms = stage_er.select_landmark_genes(gene_info, landmark_flag_col=GENE_LM_COL,
    gene_id_col=GENE_ID_COL, gene_symbol_col=GENE_SYM_COL)
gctx_path = raw / 'level5_modz.gctx'
print(f'Fetching the Level-5 gctx to slice {n_sig_ids} sig_ids x {len(gene_ids)} landmark genes '
      '(~20 GB download, ~40 GB decompressed, ~20-40 min)...')
fetch.download_gctx(loc['gse92742_level5_modz_gctx'], gctx_path, gz_path=raw / 'level5_modz.gctx.gz')
stage_er.build_lincs_parquet(sig_meta, gctx_path, gene_ids, gene_syms, staged / 'lincs.parquet')

lincs = pd.read_parquet(staged / 'lincs.parquet')
feat_cols = [c for c in lincs.columns if c != 'compound_id']
print('lincs.parquet shape :', lincs.shape, '|', len(feat_cols), 'landmark genes')
assert feat_cols == list(gene_syms), 'feature columns must be the landmark gene symbols, in order'
assert not lincs[feat_cols].isna().any().any(), 'no NaNs allowed in the landmark matrix'
print('NaNs in matrix      :', int(lincs[feat_cols].isna().sum().sum()))

In [ ]:
# 2d) GATE ONLY — approval hard-coded False here, so run_pipeline CANNOT train.
cfg = er_run.PipelineConfig.model_validate({
    'endpoint_id': 'ER', 'biological_target': 'Estrogen Receptor', 'version': '0.1.0',
    'data': {'target': 'ER', 'adapter': 'staged', 'staged_dir': 'data/staged/er',
             'n_groups': 10, 'seed': 0, 'required_label_sources': ['cerapp']},
    'gate': {'thresholds_path': 'registry/data/quality_gates.yaml'},
    'approval': {'approved': False},  # HARD BOUNDARY for Stop 2
    'evaluation': {'mode': 'nested', 'outer_splits': 5, 'inner_splits': 3},
})
res = er_run.run_pipeline(cfg, allow_list=load_sources(), data_dir=staged,
    thresholds=er_run.load_thresholds(Path('registry/data/quality_gates.yaml')), output_root=Path('.'))
assert res.trained is False  # structurally guaranteed in Stop 2
print('gate summary  :', res.gate_summary, '| overlap compounds:', res.n_overlap)
print('\n=== dataset_card.md ===\n' + Path(res.dataset_card_path).read_text())

### ⛔ STOP 2 — send Claude Chat
Send: **CERAPP labels**; **identity resolved** (+ from_inchi/fallback/unresolved split);
**matched LINCS InChIKeys**; **matched pert_ids**; **selected MCF7/A549 sig_ids**;
**pre-heavy overlap + class split**; **lincs.parquet shape + NaN check**; the **gate
summary**; and the **dataset_card.md**. Training is intentionally unreachable here. Wait
for the metric floors before Stop 3.

## STOP 3 — train (explicit human approval)
Paste the floors Claude Chat gives you into **3a-config** and set `APPROVED = True`, then
run 3b. Training runs the honest nested grouped-CV estimate + a simplicity-aware
scorecard. STOP and send the artifacts.

In [ ]:
# 3a-config) paste ONLY these values (validated_mvp floors Claude Chat provided).
FLOOR_AUROC             = 0.0   # <- paste
FLOOR_AUPRC             = 0.0   # <- paste (sized to prevalence)
FLOOR_BALANCED_ACCURACY = 0.0   # <- paste
CEIL_BRIER              = 1.0   # <- paste
APPROVED                = True  # Claude Chat authorizes training for this stop

In [ ]:
# 3b) Train: re-gates, trains (nested honest estimate), writes artifacts + proposed entry.
from pathlib import Path
from endoscan_core.datasets import load_sources
cfg = er_run.PipelineConfig.model_validate({
    'endpoint_id': 'ER', 'biological_target': 'Estrogen Receptor', 'version': '0.1.0',
    'data': {'target': 'ER', 'adapter': 'staged', 'staged_dir': 'data/staged/er',
             'n_groups': 10, 'seed': 0, 'required_label_sources': ['cerapp']},
    'gate': {'thresholds_path': 'registry/data/quality_gates.yaml'},
    'approval': {'approved': APPROVED, 'approved_by': 'operator+claude-chat'},
    'evaluation': {'mode': 'nested', 'outer_splits': 5, 'inner_splits': 3},
    'training': {'validated_mvp_floors': {'auroc': FLOOR_AUROC, 'auprc': FLOOR_AUPRC,
                                          'balanced_accuracy': FLOOR_BALANCED_ACCURACY},
                 'validated_mvp_ceilings': {'brier_score': CEIL_BRIER}},
})
res = er_run.run_pipeline(cfg, allow_list=load_sources(), data_dir=Path('data/staged/er'),
    thresholds=er_run.load_thresholds(Path('registry/data/quality_gates.yaml')), output_root=Path('.'))
print('trained:', res.trained, '| status:', res.status, '| model:', res.selected_model)
print('\n=== metrics.json ===\n' + Path('models/ER/metrics.json').read_text())
print('\n=== model_selection.md ===\n' + Path('models/ER/model_selection.md').read_text())

### ⛔ STOP 3 — send Claude Chat
`models/ER/metrics.json`, `model_selection.json`, `model_selection.md`, `model_card.md`,
`dataset_card.md`, `feature_schema.json`, and the proposed `registry/models/endpoints.json`
entry. **Do NOT push or commit yet.**

## STOP 4 — DVC-push binaries + download the review bundle (no GitHub push)
This stop **never pushes to GitHub** and **never runs git**. It DVC-pushes the binaries to
the Drive remote and downloads a small review bundle (text artifacts + `.dvc` pointers +
the proposed `endpoints.json`). You hand the bundle to Claude Code, which opens the review PR.

In [ ]:
# 4) DVC-push the BINARIES to the Drive remote (NOT GitHub), then write + download a bundle.
import zipfile
from pathlib import Path
import bundle
from google.colab import files

# (a) DVC-push ONLY the large binaries to the Google Drive local remote.
!dvc add models/ER/model.pkl data/staged/er/lincs.parquet && dvc push

# (b) Assemble the review bundle: text artifacts + small CSVs + the .dvc pointers + the
#     proposed endpoints.json. Binaries stay in DVC/Drive — they are NOT in the bundle.
REL = [
    'models/ER/metrics.json', 'models/ER/feature_schema.json',
    'models/ER/model_card.md', 'models/ER/dataset_card.md',
    'models/ER/model_selection.json', 'models/ER/model_selection.md',
    'data/staged/er/cerapp.csv', 'data/staged/er/pubchem.csv',
    'models/ER/model.pkl.dvc', 'data/staged/er/lincs.parquet.dvc',
    'registry/models/endpoints.json',
]
bundle_dir = Path('/content/drive/MyDrive/endoscan-dvc/er_phase2_bundle')
copied, missing = bundle.assemble_review_bundle(Path('.'), bundle_dir, REL)
print('bundled:', copied)
if missing:
    print('MISSING (skipped):', missing)

# (c) Zip the bundle and trigger a one-click download.
zip_path = '/content/er_phase2_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in bundle_dir.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(bundle_dir))
print('\nBundle ->', bundle_dir, '\nZipped ->', zip_path)
files.download(zip_path)

# (d) REFERENCE ONLY — do NOT run here. Hand the bundle to Claude Code (NOT in Colab).
print('\n# Reference only (Claude Code runs this locally from the bundle; NOT in Colab):')
print('git checkout -b er-real-endpoint-phase2')
print('git add models/ER/*.json models/ER/*.md models/ER/model.pkl.dvc \\')
print('        data/staged/er/lincs.parquet.dvc data/staged/er/cerapp.csv \\')
print('        data/staged/er/pubchem.csv registry/models/endpoints.json')
print('git commit -m "ER real endpoint (Phase 2): registered <status> with DVC pointers"')